
# Phân tích dữ liệu Tối ưu hóa Vận hành F&B
Đề tài: ỨNG DỤNG PHÂN TÍCH DỮ LIỆU ĐỂ TỐI ƯU HÓA QUY TRÌNH VẬN HÀNH TRONG NGÀNH F&B

Dữ liệu bao gồm 4 bảng:
1. `orders`: Thông tin thời gian đặt hàng
2. `order_details`: Chi tiết từng đơn (loại pizza, số lượng)
3. `pizzas`: Thông tin về size và giá của từng loại pizza cụ thể
4. `pizza_types`: Thông tin về danh mục và thành phần nguyên liệu


In [ ]:

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style cho biểu đồ đẹp hơn
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Tải dữ liệu
order_details = pd.read_csv('order_details.csv')
orders = pd.read_csv('orders.csv')
pizza_types = pd.read_csv('pizza_types.csv', encoding='latin1')
pizzas = pd.read_csv('pizzas.csv')



## 1. Tiền xử lý và Gộp Dữ liệu (Data Preprocessing)
Mục tiêu là tạo ra một bảng dữ liệu tổng hợp (master dataset) chứa đầy đủ thông tin về doanh thu, thời gian và chi tiết sản phẩm.


In [ ]:

# Gộp order_details và pizzas (Lấy giá và size)
df_merged = pd.merge(order_details, pizzas, on='pizza_id', how='left')

# Gộp với pizza_types (Lấy tên, danh mục, nguyên liệu)
df_merged = pd.merge(df_merged, pizza_types, on='pizza_type_id', how='left')

# Gộp với orders (Lấy thời gian đặt hàng)
df_merged = pd.merge(df_merged, orders, on='order_id', how='left')

# Tính doanh thu của từng dòng (Revenue = quantity * price)
df_merged['revenue'] = df_merged['quantity'] * df_merged['price']

# Xử lý thời gian (Chuyển date và time sang định dạng Datetime)
df_merged['date'] = pd.to_datetime(df_merged['date'])
df_merged['time'] = pd.to_datetime(df_merged['time'], format='%H:%M:%S').dt.time
df_merged['hour'] = df_merged['time'].apply(lambda x: x.hour)
df_merged['day_of_week'] = df_merged['date'].dt.day_name()

display(df_merged.head())



### 1.2 Feature Engineering (Tạo đặc trưng mới)
Tạo các biến hữu ích phục vụ phân tích chi tiết hơn: Ngày cuối tuần, Buổi trong ngày, và Phân khúc quy mô đơn hàng.


In [ ]:

df_merged['is_weekend'] = df_merged['day_of_week'].isin(['Saturday', 'Sunday'])

def get_part_of_day(hour):
    if 5 <= hour < 11: return 'Sáng'
    elif 11 <= hour < 14: return 'Trưa'
    elif 14 <= hour < 17: return 'Chiều'
    elif 17 <= hour < 21: return 'Tối'
    else: return 'Khuya'
df_merged['part_of_day'] = df_merged['hour'].apply(get_part_of_day)

order_totals = df_merged.groupby('order_id')['revenue'].sum().reset_index()
order_totals.rename(columns={'revenue': 'total_order_value'}, inplace=True)
def get_order_size(val):
    if val < 20: return 'Nhỏ'
    elif val < 50: return 'Vừa'
    else: return 'Lớn'
order_totals['order_size'] = order_totals['total_order_value'].apply(get_order_size)

df_merged = pd.merge(df_merged, order_totals, on='order_id', how='left')
display(df_merged[['order_id', 'date', 'part_of_day', 'is_weekend', 'total_order_value', 'order_size']].head())



## 2. Tối ưu hóa Nhân sự (Staff Scheduling Optimization)
Phân tích thời gian đặt hàng để tìm ra khung giờ và ngày cao điểm, giúp quản lý phân bổ nhân viên hợp lý.


In [ ]:

# 2.1 Số lượng đơn hàng theo Giờ trong ngày
orders_by_hour = df_merged.groupby('hour')['order_id'].nunique().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=orders_by_hour, x='hour', y='order_id', palette='viridis')
plt.title('Số lượng đơn hàng theo Giờ trong ngày', fontsize=16)
plt.xlabel('Giờ', fontsize=12)
plt.ylabel('Số lượng Đơn hàng', fontsize=12)
plt.xticks(range(0, len(orders_by_hour['hour'])))
plt.show()


In [ ]:

# 2.2 Số lượng đơn hàng theo Thứ trong tuần
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
orders_by_day = df_merged.groupby('day_of_week')['order_id'].nunique().reindex(day_order).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=orders_by_day, x='day_of_week', y='order_id', palette='mako')
plt.title('Số lượng đơn hàng theo Thứ trong tuần', fontsize=16)
plt.xlabel('Thứ', fontsize=12)
plt.ylabel('Số lượng Đơn hàng', fontsize=12)
plt.show()



## 3. Tối ưu hóa Thực đơn (Menu Engineering)
Phân tích xem loại pizza và size nào mang lại doanh thu lớn nhất để điều chỉnh menu và các chương trình khuyến mãi.


In [ ]:

# 3.1 Top 10 Pizza mang lại Doanh thu cao nhất
revenue_by_pizza = df_merged.groupby('name')['revenue'].sum().sort_values(ascending=False).head(10).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=revenue_by_pizza, y='name', x='revenue', palette='magma')
plt.title('Top 10 Pizza có Doanh thu cao nhất', fontsize=16)
plt.xlabel('Tổng Doanh thu ($)', fontsize=12)
plt.ylabel('Tên Pizza', fontsize=12)
plt.show()


In [ ]:

# 3.2 Tỷ trọng doanh thu theo Kích cỡ (Size)
revenue_by_size = df_merged.groupby('size')['revenue'].sum().reset_index()

plt.figure(figsize=(8, 8))
plt.pie(revenue_by_size['revenue'], labels=revenue_by_size['size'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Tỷ trọng Doanh thu theo Kích cỡ (Size)', fontsize=16)
plt.show()



## 4. Tối ưu hóa Kho nguyên liệu (Inventory Management)
Phân tích các nguyên liệu được tiêu thụ nhiều nhất để lên kế hoạch nhập hàng và tránh lãng phí.


In [ ]:

# 4.1 Tách các nguyên liệu ra khỏi cột 'ingredients' và tính tổng số lượng tiêu thụ
from collections import Counter

# Đếm nguyên liệu theo số lượng pizza đã bán (quantity)
ingredient_counts = Counter()
for index, row in df_merged.dropna(subset=['ingredients']).iterrows():
    ingredients_list = [i.strip() for i in row['ingredients'].split(',')]
    for ing in ingredients_list:
        ingredient_counts[ing] += row['quantity']

# Chuyển thành Dataframe và lấy top 15
top_ingredients = pd.DataFrame.from_dict(ingredient_counts, orient='index', columns=['total_consumed']).sort_values('total_consumed', ascending=False).head(15).reset_index()
top_ingredients.rename(columns={'index': 'ingredient'}, inplace=True)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_ingredients, y='ingredient', x='total_consumed', palette='cubehelix')
plt.title('Top 15 Nguyên liệu tiêu thụ nhiều nhất (Theo số lượng món)', fontsize=16)
plt.xlabel('Số lượng', fontsize=12)
plt.ylabel('Nguyên liệu', fontsize=12)
plt.show()



## 5. Phân tích Giỏ hàng (Market Basket Analysis - Cross-Selling)
Tìm các tổ hợp sản phẩm thường được khách hàng mua cùng nhau trong một đơn.


In [ ]:

from itertools import combinations
from collections import Counter

# Lấy ra danh sách sản phẩm trong mỗi đơn hàng
basket = df_merged.groupby('order_id')['name'].apply(list)
# Chỉ xét các đơn mua >= 2 sản phẩm
basket = basket[basket.apply(len) >= 2]

pair_counter = Counter()
for items in basket:
    pair_counter.update(combinations(sorted(items), 2))

# In ra top 10 cặp mua chung nhiều nhất
top_10_pairs = pair_counter.most_common(10)
pair_df = pd.DataFrame(top_10_pairs, columns=['Cặp Pizza (Pair)', 'Số lần mua chung (Frequency)'])
pair_df['Cặp Pizza (Pair)'] = pair_df['Cặp Pizza (Pair)'].apply(lambda x: f"{x[0]} & {x[1]}")

plt.figure(figsize=(10, 6))
sns.barplot(data=pair_df, y='Cặp Pizza (Pair)', x='Số lần mua chung (Frequency)', palette='rocket')
plt.title('Top 10 Cặp Pizza thường được mua chung', fontsize=14)
plt.show()



## 6. Phát hiện điểm dị biệt (Outlier Detection)
Phát hiện những ngày có doanh thu tăng đột biến (ví dụ: ngày lễ, sự kiện) sử dụng phương pháp Interquartile Range (IQR).


In [ ]:

# Tính tổng doanh thu theo ngày
daily_rev = df_merged.groupby('date')['revenue'].sum().reset_index()

# Tính IQR
Q1 = daily_rev['revenue'].quantile(0.25)
Q3 = daily_rev['revenue'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Xác định Outliers
outliers = daily_rev[daily_rev['revenue'] > upper_bound]

plt.figure(figsize=(12, 6))
sns.scatterplot(data=daily_rev, x='date', y='revenue', color='blue', label='Doanh thu bình thường')
if not outliers.empty:
    sns.scatterplot(data=outliers, x='date', y='revenue', color='red', s=100, label='Doanh thu đột biến (Outlier)')
    
plt.axhline(y=upper_bound, color='r', linestyle='--', label=f'Ngưỡng cảnh báo (Upper Bound = {upper_bound:.2f})')
plt.title('Phát hiện Doanh thu Dị biệt theo Ngày', fontsize=16)
plt.xlabel('Ngày')
plt.ylabel('Doanh thu ($)')
plt.legend()
plt.show()

display(outliers)
